# Classifier Training and InspectionThe feature extraction and training code now lives in `src/uno_vision/classification/`.This notebook focuses on the sample set, feature shapes, model metrics, and quick prediction previews.

In [ ]:
import sysfrom collections import Counterfrom pathlib import PathPROJECT_ROOT = Path.cwd().resolve()while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:    PROJECT_ROOT = PROJECT_ROOT.parentif str(PROJECT_ROOT / "src") not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT / "src"))import cv2import matplotlib.pyplot as pltimport numpy as npfrom uno_vision.classification.features import (    IMG_SIZE,    extract_color_features,    extract_rank_features,    letterbox_cv2,    split_card_label,)from uno_vision.classification.predict import load_card_classifierfrom uno_vision.classification.training import (    collect_classifier_samples,    save_classifier_artifacts,    train_classifiers,)from uno_vision.paths import CLASSIFIER_MODELS_DIRprint(f"Project root: {PROJECT_ROOT}")print(f"Classifier artifacts: {CLASSIFIER_MODELS_DIR}")

In [ ]:
samples = collect_classifier_samples()labels = [label for _, label in samples]color_counts = Counter()rank_counts = Counter()for label in labels:    color, rank = split_card_label(label)    color_counts[color] += 1    rank_counts[rank] += 1print(f"Samples: {len(samples)}")print(f"Unique full labels: {len(set(labels))}")print(f"Unique colors: {len(color_counts)}")print(f"Unique ranks: {len(rank_counts)}")

In [ ]:
def plot_counter(ax, title, counter):    items = sorted(counter.items(), key=lambda item: (-item[1], item[0]))    names = [item[0] for item in items]    values = [item[1] for item in items]    ax.barh(range(len(names)), values)    ax.set_yticks(range(len(names)))    ax.set_yticklabels(names)    ax.invert_yaxis()    ax.set_title(title)fig, axes = plt.subplots(1, 2, figsize=(14, 6))plot_counter(axes[0], "Color label counts", color_counts)plot_counter(axes[1], "Rank label counts", rank_counts)plt.tight_layout()plt.show()

In [ ]:
sample_path, sample_label = samples[0]img_bgr = cv2.imread(sample_path, cv2.IMREAD_COLOR)img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)img_lb = letterbox_cv2(img_bgr, size=IMG_SIZE)img_lb_rgb = cv2.cvtColor(img_lb, cv2.COLOR_BGR2RGB)color_features = extract_color_features(img_lb)rank_features = extract_rank_features(img_lb)fig, axes = plt.subplots(1, 2, figsize=(8, 4))axes[0].imshow(img_rgb)axes[0].set_title(f"Original\n{sample_label}")axes[0].axis("off")axes[1].imshow(img_lb_rgb)axes[1].set_title("Letterboxed for features")axes[1].axis("off")plt.tight_layout()plt.show()print(f"Sample path: {sample_path}")print(f"Color feature length: {color_features.shape[0]}")print(f"Rank feature length:  {rank_features.shape[0]}")

In [ ]:
result = train_classifiers(samples=samples, test_size=0.2, random_state=42)for key, value in result.metrics.items():    print(f"{key}: {value:.4f}")

In [ ]:
if "result" not in globals():    raise RuntimeError("Run the training cell first so 'result' is available.")save_classifier_artifacts(result)print(f"Saved classifier artifacts to: {CLASSIFIER_MODELS_DIR}")

In [ ]:
classifier = load_card_classifier()step = max(1, len(samples) // 8)preview_samples = samples[::step][:8]fig, axes = plt.subplots(2, 4, figsize=(14, 7))axes = axes.reshape(-1)for ax, (path, true_label) in zip(axes, preview_samples):    img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)    pred = classifier.predict_bgr(img_bgr)    ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))    ax.set_title(f"T: {true_label}\nP: {pred.label}", fontsize=8)    ax.axis("off")for ax in axes[len(preview_samples):]:    ax.axis("off")plt.tight_layout()plt.show()